# Gauss-Legendre check (Python)

Checks `thole.quad.GaussLegendre` and `gauss_legendre` against closed forms, mpmath, SciPy, and what the Lean theorems predict.
Start JupyterLab with `workshop run -- notebook`. After changing the Rust code, restart it.

In [ ]:
import math
import mpmath
import numpy as np
from scipy import integrate, special
import matplotlib.pyplot as plt
from thole.quad import GaussLegendre, gauss_legendre

## 1. The rule with 3 points

The nodes are `0, ±sqrt(3/5)` and the weights `8/9, 5/9`.

In [ ]:
rule = GaussLegendre(3)
print(rule, rule.nodes, rule.weights)
print("node error  :", [x - e for x, e in zip(rule.nodes, [-math.sqrt(0.6), 0.0, math.sqrt(0.6)])])
print("weight error:", [w - e for w, e in zip(rule.weights, [5 / 9, 8 / 9, 5 / 9])])
print("hex (compare with the Rust notebook):", rule.nodes[2].hex(), rule.weights[2].hex())

## 2. Nodes and weights against mpmath and SciPy

`scipy.special.roots_legendre` computes the same rule independently. Errors are in units of `2^-53`.

In [ ]:
mpmath.mp.dps = 50
for n in [5, 20, 64]:
    rule = GaussLegendre(n)
    x_s, w_s = special.roots_legendre(n)
    exact = sorted(mpmath.polyroots(mpmath.taylor(lambda t: mpmath.legendre(n, t), 0, n)[::-1], maxsteps=500, extraprec=1000))
    ulp = lambda d: float(abs(d) / mpmath.mpf(2) ** -53)
    print(f"n={n:3}  thole: largest node error {max(ulp(mpmath.mpf(x) - e) for x, e in zip(rule.nodes, exact)):.2f}"
          f"   scipy: {max(ulp(mpmath.mpf(float(x)) - e) for x, e in zip(x_s, exact)):.2f}"
          f"   largest weight difference from scipy {max(abs(a - b) for a, b in zip(rule.weights, w_s)):.2e}")

## 3. What the Lean theorems predict

`gauss_exact` and `rule_exact_of_exact`: the rule with `n` points integrates every polynomial of degree at most `2n - 1` exactly, on any interval; degree `2n` is the first that it misses.

In [ ]:
a, b = -0.5, 2.0
print(f"{'n':>3}{'degree 2n-1 error':>20}{'degree 2n error':>18}")
for n in range(1, 11):
    rule = GaussLegendre(n)
    errs = []
    for d in (2 * n - 1, 2 * n):
        exact = (b ** (d + 1) - a ** (d + 1)) / (d + 1)
        errs.append(abs(rule.integrate(lambda x: x**d, a, b) - exact) / abs(exact))
    print(f"{n:3}{errs[0]:20.2e}{errs[1]:18.2e}")

## 4. Convergence on smooth and on singular integrands

For an analytic integrand the error falls exponentially with `n`; for `sqrt(x)` on `[0, 1]`, whose derivative is infinite at 0, only algebraically.

In [ ]:
ns = list(range(1, 41))
for name, f, a, b, exact in [
    ("exp(x)", math.exp, 0, 1, math.e - 1),
    ("1/(1+25x^2)", lambda x: 1 / (1 + 25 * x * x), -1, 1, 2 * math.atan(5) / 5),
    ("sqrt(x)", math.sqrt, 0, 1, 2 / 3),
]:
    errors = [max(abs(gauss_legendre(f, a, b, n) - exact), 1e-17) for n in ns]
    plt.semilogy(ns, errors, "o-", label=name)
plt.xlabel("n"); plt.ylabel("|error|"); plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()

## 5. Against SciPy's `fixed_quad`

In [ ]:
for name, f, a, b in [("exp", np.exp, 0.0, 1.0), ("cos 30x", lambda x: np.cos(30 * x), 0.0, 1.0)]:
    for n in [5, 20, 40]:
        s, _ = integrate.fixed_quad(f, a, b, n=n)
        t = gauss_legendre(lambda x: float(f(x)), a, b, n)
        print(f"{name:8} n={n:3}  thole={t:.17g}  scipy={s:.17g}  difference={abs(t - s):.1e}")

## 6. Failures

In [ ]:
for name, call in {
    "zero points":     lambda: GaussLegendre(0),
    "infinite end":    lambda: gauss_legendre(math.exp, 0, math.inf, 5),
    "f returns inf":   lambda: gauss_legendre(lambda x: 1 / x if x else math.inf, -1, 1, 5),
    "f raises":        lambda: gauss_legendre(lambda x: 1 / 0, 0, 1, 5),
}.items():
    try:
        print(f"{name:16} -> returned {call()!r}")
    except Exception as e:
        print(f"{name:16} -> {type(e).__name__}: {e}")